# Part D — Multi-Country Interconnected Network (DC Approximation)

Countries: Denmark (DNK), Sweden (SWE), Norway (NOR), Germany (DEU)  
Lines: 400 kV, x=0.1, fixed capacities from existing infrastructure.

1. **Section 1 (Optimisation)** — run once, saves results to `plots_part_d/`
2. **Section 2 (Plotting)** — loads from CSV, re-run freely without the solver

---
## Section 1 — Optimisation
*Run these cells once. Skip after CSVs exist.*

In [ ]:
import os
import pandas as pd
import pypsa

df_elec = pd.read_csv('data/electricity_demand.csv', sep=';', index_col=0)
df_elec.index = pd.to_datetime(df_elec.index)
df_onshorewind = pd.read_csv('data/onshore_wind_1979-2017.csv', sep=';', index_col=0)
df_onshorewind.index = pd.to_datetime(df_onshorewind.index)
df_offshorewind = pd.read_csv('data/offshore_wind_1979-2017.csv', sep=';', index_col=0)
df_offshorewind.index = pd.to_datetime(df_offshorewind.index)
df_solar = pd.read_csv('data/pv_optimal.csv', sep=';', index_col=0)
df_solar.index = pd.to_datetime(df_solar.index)

countries = ["DNK", "SWE", "NOR", "DEU"]

def annuity(n, r):
    return r / (1. - 1. / (1. + r) ** n) if r > 0 else 1 / n

In [ ]:
network = pypsa.Network()
hours_in_2015 = pd.date_range('2015-01-01 00:00Z', '2015-12-31 23:00Z', freq='h')
network.set_snapshots(hours_in_2015.values)

for c in countries:
    network.add("Bus", c, v_nom=400)
    network.add("Load", f"load_{c}", bus=c, p_set=df_elec[c].values)

def add_country_generators(network, country):
    for carrier in ["gas", "onshorewind", "offshorewind", "solar", "battery storage", "pumped hydro"]:
        if carrier not in network.carriers.index:
            network.add("Carrier", carrier, co2_emissions=0.19 if carrier == "gas" else 0)

    CF_on  = df_onshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
    CF_off = df_offshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
    CF_pv  = df_solar[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]

    network.add("Generator", f"Onshore wind {country}", bus=country, carrier="onshorewind",
                p_nom_extendable=True, capital_cost=annuity(27, 0.07) * 1118775,
                marginal_cost=0, p_max_pu=CF_on.values)
    network.add("Generator", f"Offshore wind {country}", bus=country, carrier="offshorewind",
                p_nom_extendable=True, capital_cost=annuity(27, 0.07) * 2115944,
                marginal_cost=0, p_max_pu=CF_off.values)
    network.add("Generator", f"Solar {country}", bus=country, carrier="solar",
                p_nom_extendable=True, capital_cost=annuity(25, 0.07) * 450000,
                marginal_cost=0, p_max_pu=CF_pv.values)
    network.add("Generator", f"OCGT {country}", bus=country, carrier="gas",
                p_nom_extendable=True, capital_cost=annuity(25, 0.07) * 453960,
                marginal_cost=30 / 0.41)
    network.add("Generator", f"CCGT {country}", bus=country, carrier="gas",
                p_nom_extendable=True, capital_cost=annuity(25, 0.07) * 880000,
                marginal_cost=30 / 0.56)
    network.add("StorageUnit", f"battery storage {country}", bus=country, carrier="battery storage",
                p_nom_extendable=True, max_hours=2,
                capital_cost=annuity(20, 0.07) * 2 * 288000,
                efficiency_store=0.98, efficiency_dispatch=0.97,
                cyclic_state_of_charge=True)

for c in countries:
    add_country_generators(network, c)

In [ ]:
def load_hydro_inflow(csv_path, country_code, year=2012):
    df = pd.read_csv(csv_path)
    df = df[df["Year"] == year].copy()
    df['date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
    return df.set_index('date')['Inflow [GWh]'].reindex(network.snapshots).fillna(0).values

for country, csv, p_nom_max in [
    ("SWE", 'data/inflow/Hydro_Inflow_SE.csv', 16000),
    ("NOR", 'data/inflow/Hydro_Inflow_NO.csv', 33000),
    ("DEU", 'data/inflow/Hydro_Inflow_DE.csv',  7000),
]:
    network.add("StorageUnit", f"pumped hydro {country}",
                bus=country, carrier="pumped hydro",
                p_nom_extendable=True, p_nom_max=p_nom_max, max_hours=8,
                capital_cost=annuity(80, 0.07) * 400000,
                efficiency_store=0.9, efficiency_dispatch=0.9,
                cyclic_state_of_charge=True, marginal_cost=1,
                inflow=load_hydro_inflow(csv, country))

x_line = 0.1
for name, b0, b1, s_nom in [
    ("DK-NO", "DNK", "NOR", 1632),
    ("DK-SE", "DNK", "SWE", 2415),
    ("DK-DE", "DNK", "DEU", 3500),
    ("SE-NO", "SWE", "NOR", 3945),
    ("SE-DE", "SWE", "DEU",  615),
    ("NO-DE", "NOR", "DEU", 1400),
]:
    network.add("Line", name, bus0=b0, bus1=b1, x=x_line, s_nom=s_nom)

In [ ]:
network.optimize(solver_name="gurobi", solver_options={"OutputFlag": 0, "LogToConsole": 0})
print(f"Total system cost: {network.objective / 1e6:.2f} M\u20ac/yr")
print(f"CO\u2082 shadow price: {network.global_constraints.mu.get('co2_limit', 'N/A')}")

In [ ]:
import re
os.makedirs('plots_part_d', exist_ok=True)

optimal_cap = pd.concat([network.generators.p_nom_opt, network.storage_units.p_nom_opt]) / 1000
cap_table = (
    optimal_cap.rename_axis("name").reset_index(name="capacity")
    .assign(country=lambda df: df["name"].str.split().str[-1],
            tech=lambda df: df["name"].str.replace(r"\s+(DNK|SWE|NOR|DEU)$", "", regex=True))
    .pivot(index="country", columns="tech", values="capacity")
    .reindex(index=["DNK", "SWE", "NOR", "DEU"],
             columns=["Onshore wind", "Offshore wind", "Solar", "OCGT", "CCGT",
                      "battery storage", "pumped hydro"])
    .fillna(0)
    .rename(index={"DNK": "Denmark", "SWE": "Sweden", "NOR": "Norway", "DEU": "Germany"})
)
cap_table.to_csv('plots_part_d/cap_table.csv')

gen_by_name = network.generators_t.p.sum() / 1e6
gen_by_name.index = gen_by_name.index.str.replace(r"\s+(DNK|SWE|NOR|DEU)$", "", regex=True)
storage_by_name = network.storage_units_t.p.clip(lower=0).sum() / 1e6
storage_by_name.index = storage_by_name.index.str.replace(
    r"(battery storage|pumped hydro)\s+(DNK|SWE|NOR|DEU)$", r"\1", regex=True)
generation_mix = pd.concat([gen_by_name.groupby(level=0).sum(),
                             storage_by_name.groupby(level=0).sum()])
generation_mix = generation_mix.reindex(
    ["Onshore wind", "Offshore wind", "Solar", "OCGT", "CCGT", "battery storage", "pumped hydro"]
).fillna(0)
generation_mix.to_csv('plots_part_d/generation_mix.csv')

# Dispatch by tech for duration curves
gen_dispatch = network.generators_t.p.copy()
gen_dispatch.columns = gen_dispatch.columns.str.replace(r"\s+(DNK|SWE|NOR|DEU)$", "", regex=True)
storage_dispatch = network.storage_units_t.p.copy()
storage_dispatch.columns = storage_dispatch.columns.str.replace(
    r"(battery storage|pumped hydro)\s+(DNK|SWE|NOR|DEU)$", r"\1", regex=True)
dispatch_by_tech = pd.concat([gen_dispatch.T.groupby(level=0).sum().T,
                               storage_dispatch.T.groupby(level=0).sum().T], axis=1)
dispatch_by_tech = dispatch_by_tech.T.groupby(level=0).sum().T
dispatch_by_tech.to_csv('plots_part_d/dispatch_by_tech.csv')

print("Saved CSVs to plots_part_d/")
print(cap_table)

---
## Section 2 — Plotting
*Start here after CSVs exist. No solver needed.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cap_table      = pd.read_csv('plots_part_d/cap_table.csv', index_col=0)
generation_mix = pd.read_csv('plots_part_d/generation_mix.csv', index_col=0, header=None).squeeze()
dispatch       = pd.read_csv('plots_part_d/dispatch_by_tech.csv', index_col=0, parse_dates=True)

TECH_COLS   = ["Onshore wind", "Offshore wind", "Solar", "OCGT", "CCGT", "battery storage", "pumped hydro"]
TECH_LABELS = ['Onshore wind', 'Offshore wind', 'Solar PV', 'Gas (OCGT)', 'Gas (CCGT)', 'Battery storage', 'Hydro']
COLORS      = ['blue', 'dodgerblue', 'orange', 'crimson', 'darkviolet', 'lightgreen', 'pink']

print("Installed capacities [GW]:")
cap_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
cap_table[TECH_COLS].plot(kind="bar", stacked=True, colors=COLORS, rot=0, ax=ax)
ax.set_ylabel("Installed capacity [GW]")
ax.set_xlabel("")
ax.legend(TECH_LABELS, bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.savefig('plots_part_d/1d_capacities.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
total_cap = cap_table[TECH_COLS].sum()
perc = 100 * total_cap / total_cap.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

ax1.pie(total_cap, colors=COLORS,
        labels=[f'{l}\n{p:.1f}%' for l, p in zip(TECH_LABELS, perc)],
        startangle=90, labeldistance=1.18)
ax1.set_title('Installed capacity mix', fontweight='bold')

gen_vals = generation_mix.reindex(TECH_COLS).fillna(0)
perc_gen = 100 * gen_vals / gen_vals.sum()
ax2.pie(gen_vals, colors=COLORS,
        labels=[f'{l}\n{p:.1f}%' for l, p in zip(TECH_LABELS, perc_gen)],
        startangle=90, labeldistance=1.18, wedgeprops={'linewidth': 0})
ax2.set_title('Annual generation mix', fontweight='bold')

plt.tight_layout()
plt.savefig('plots_part_d/1d_mix_pies.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5), dpi=150)
for tech, label, color in zip(TECH_COLS, TECH_LABELS, COLORS):
    if tech in dispatch.columns:
        sorted_d = dispatch[tech].sort_values(ascending=False).reset_index(drop=True)
        plt.plot(sorted_d.values, label=label, color=color, lw=2.5)
plt.axhline(0, color='black', lw=0.8)
plt.ylabel('Dispatch [MWh/h]')
plt.xlabel('Hours')
plt.title('Duration curve of generation and storage operation')
plt.legend(fancybox=True, shadow=True, loc='best', fontsize=8)
plt.tight_layout()
plt.savefig('plots_part_d/1d_duration_curve.png', dpi=300, bbox_inches='tight')
plt.show()